# Step 6: Review real terms before calculating costs

Select **.venv**, restart the kernel after installing the updated code, and run
from the top. This lesson loads your saved Reliant extraction. No API calls are made.

**Expected first outcome: blocked.** Passing citation checks is different from
having complete pricing rules. We will keep unknown charges unknown and create an
editable review document. No one has approved your plan by running this notebook.

## 1. Load source evidence
Read the real PDF with table cells, and load its saved workflow result. Recheck
citations rather than assuming old validation flags are current. Supplementary
pricing documents can be added to `extra_source_paths` later.

In [ ]:
from pathlib import Path
from decimal import Decimal
import json
from IPython.display import display, HTML
from html import escape
from electricity_optimizer.ingestion import read_pdf
from electricity_optimizer.agreements import ExtractionRecord
from electricity_optimizer.extraction import validate_terms
from electricity_optimizer.review import (
    PricingReview, ReviewedValue, Evidence, review_blockers, compile_reviewed_plan,
    SCENARIO_ASSUMPTIONS, SUPPORTED_STRUCTURE,
)
from electricity_optimizer.usage import load_usage_csv
from electricity_optimizer.pricing import calculate_month

PROJECT_ROOT = Path.cwd()
workflow_path = PROJECT_ROOT / "output" / "workflow" / "R1F00169972621A_workflow.json"
if not workflow_path.is_file():
    raise FileNotFoundError("Save the Reliant workflow report in Notebook 03 first.")
workflow_report = json.loads(workflow_path.read_text(encoding="utf-8"))
record = ExtractionRecord.model_validate(workflow_report["record"])
document = read_pdf(PROJECT_ROOT / "contracts" / "R1F00169972621A.pdf", include_tables=True)
if record.source_sha256 != document.sha256:
    raise ValueError("Saved extraction does not match this PDF.")
extra_source_paths = []  # Add local matching pricing-document Paths when obtained.
documents = [document] + [read_pdf(path, include_tables=True) for path in extra_source_paths]
for issue in validate_terms(record.terms, document):
    print(issue.severity, issue.field, issue.message)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
warning customer_type Not stated: obtain supporting documents before relying on this term.
warning recurring_charges Not stated: obtain supporting documents before relying on this term.
warning credits_and_minimum_usage Not stated: obtain supporting documents before relying on this term.
warning renewal_terms Not stated: obtain supporting documents before relying on this term.


## 2. Inspect what the model actually found
Start with energy charges, delivery charges and recurring fees. A pass-through
condition is not a numeric rate. An illustrative average price is not the same
as the unit energy charge. Missing credits do not establish that no credits exist.

In [ ]:
for name in ["plan_name", "market", "energy_charges", "average_prices", "delivery_charges",
             "recurring_charges", "credits_and_minimum_usage", "price_change_rules"]:
    term = getattr(record.terms, name)
    print(f"\n{name} [{term.status}]: {term.value}")
    for citation in term.citations:
        print(f"  Page {citation.page_number}: {citation.quote}")


plan_name [found]: Reliant Plain and Simple 36 plan
  Page 1: Reliant Plain and Simple 36 plan

market [found]: AEP Texas Central service area
  Page 1: AEP Texas Central service area

energy_charges [found]: Energy Charge: 18.2499¢ per kWh
  Page 1: Energy Charge:  18.2499¢  per kWh

average_prices [found]: Average price per kWh at average monthly use: 500 kWh – 18.2¢; 1000 kWh – 18.2¢; 2000 kWh – 18.2¢
  Page 1: Average monthly use:      500 kWh         1000 kWh         2000 kWh
  Page 1: Average price per kWh:          18.2 ¢              18.2 ¢              18.2 ¢

delivery_charges [found]: AEP Texas Central Delivery Charges are passed through without mark-up; customers in the McAllen/Mission area formerly served by Oncor will not be assessed TC-2, TC-3, NDC, or SRC charges or the ADFIT credit associated with the SRC.
  Page 1: AEP Texas Central Delivery Charges include all recurring charges from AEP Texas Central passed through
               without mark-up.
  Page 1: **Customer

## 3. Understand the supported calculation scope
This first real-input adapter supports a **flat USD rate with no credits, tiers,
time-of-use periods or minimum bills**. Plans outside that subset stay blocked.
It reuses the monthly arithmetic from Notebook 04 without calling a real plan fictional.

We only calculate a hypothetical energy/base/delivery subtotal under constant rates.
Actual taxes, switching charges, renewal periods, and changing delivery tariffs
are not covered. Matching quotes does not prove numerical conversion, territory
eligibility or completeness: the reviewer must check those against the documents.

In [ ]:
print(SCENARIO_ASSUMPTIONS)
print("Supported pricing_structure value:", SUPPORTED_STRUCTURE)
# Unit conversion exercise only, not an approved pricing input:
print("18.2499 cents/kWh in dollars/kWh:", Decimal("18.2499") / Decimal("100"))

Hypothetical 12-month scenario with constant reviewed rates; not a current offer or forecast. Energy, base and delivery only; taxes, switching/termination charges and deposits excluded. Round each component to cents half up. No tiers, time-of-use, credits or minimum bills.
Supported pricing_structure value: flat_no_credits_no_tiers_no_minimum_bill
18.2499 cents/kWh in dollars/kWh: 0.182499


## 4. Review all pricing inputs in one form
The setup cell preserves your existing JSON and completed reviews. Then run the
form cell below. Expand a field to see its value and citations together.

- Enter your name once and verify the review date.
- The energy rate may be prefilled from an exact cited cents/kWh charge, divided
  by 100. It remains a candidate until you check the conditions and evidence.
- Tick **I checked this value and its evidence** only for fields you actually reviewed.
- Leave unknown charges blank. Use **Add citation** to select a loaded source,
  page and exact quote when you obtain supporting information.
- Click **Save review & check**. Saving a draft does not require approving every field.

Your previous reviews are preserved when unchanged. Editing a value or evidence
clears its old review unless you explicitly review the new version. Saving creates
a backup of the prior JSON. If the JSON changed outside the form, reopen the form
cell to load those edits. No API calls occur.

If widgets do not load, select .venv and restart the kernel. Dependencies are in
`requirements-notebook.txt`; the form uses ipywidgets.

In [ ]:
review_dir = PROJECT_ROOT / "output" / "review"
review_dir.mkdir(parents=True, exist_ok=True)
review_path = review_dir / "Reliant_pricing_review.json"
if not review_path.exists():
    draft = PricingReview(source_file=document.source_file, source_sha256=document.sha256)
    for field, extracted in [("plan_name", record.terms.plan_name), ("territory", record.terms.market)]:
        setattr(draft, field, ReviewedValue(value=extracted.value,
            evidence=[Evidence(source_sha256=document.sha256, page_number=q.page_number, quote=q.quote)
                      for q in extracted.citations]))
    review_path.write_text(draft.model_dump_json(indent=2), encoding="utf-8")
print("Review file:", review_path)
print("Available sources:")
for source in documents:
    print(source.source_file, source.sha256)

Review file: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\review\Reliant_pricing_review.json
Available sources:
R1F00169972621A.pdf 6f88df972b9d1cc41a06e036502f5e81a6f955644e79fa696f5b275b1c3bc56b


In [ ]:
from electricity_optimizer.review_form import show_review_form

review_form = show_review_form(review_path, record, documents)

## 5. Check saved readiness
Click **Save review & check** in the form first, then rerun this cell. Unsaved form
edits are not used for calculation. The output groups missing information by field.
Currency and pricing structure remain explicit requirements; do not approve them
solely because a value is accepted by the program.

In [ ]:
review = PricingReview.model_validate_json(review_path.read_text(encoding="utf-8"))
blockers = review_blockers(review, documents)
print("BLOCKED" if blockers else "Ready for the limited hypothetical calculation")
for field in PricingReview.model_fields:
    item = getattr(review, field)
    if isinstance(item, ReviewedValue):
        issues = [b.split(": ", 1)[1] for b in blockers if b.startswith(field + ":")]
        print(f"\n{field}: {item.value if item.value is not None else 'unknown'}")
        print("  " + ("; ".join(issues) if issues else "Review and citation checks recorded"))
for blocker in blockers:
    if not any(blocker.startswith(field + ":") for field in PricingReview.model_fields):
        print("\n" + blocker)

BLOCKED

plan_name: Reliant Plain and Simple 36 plan
  Review and citation checks recorded

territory: AEP Texas Central service area
  Review and citation checks recorded

currency: USD
  Review and citation checks recorded

energy_usd_per_kwh: 0.182499
  Review and citation checks recorded

base_usd_per_month: unknown
  missing value.; human review not recorded.; record review date as YYYY-MM-DD.; source evidence missing (including for zero charges).

delivery_usd_per_kwh: unknown
  missing value.; human review not recorded.; record review date as YYYY-MM-DD.; source evidence missing (including for zero charges).

delivery_usd_per_month: unknown
  missing value.; human review not recorded.; record review date as YYYY-MM-DD.; source evidence missing (including for zero charges).

pricing_structure: unknown
  missing value.; human review not recorded.; record review date as YYYY-MM-DD.; source evidence missing (including for zero charges).

Confirm the supported flat-rate structure; ot

## 6. Guard the calculation
This cell reloads the review before compiling it, so stale notebook variables cannot
approve a changed file. With the current draft it prints BLOCKED and no annual cost.
Once evidence and review are complete, it can apply the supported plan to Notebook
04's synthetic usage. It does not rank this plan against fictional offers.

In [ ]:
review = PricingReview.model_validate_json(review_path.read_text(encoding="utf-8"))
blockers = review_blockers(review, documents)
reviewed_plan = None
reviewed_bills = None
if blockers:
    print("BLOCKED: resolve the listed evidence and review requirements first.")
else:
    reviewed_plan = compile_reviewed_plan(review, documents)
    usage_path = PROJECT_ROOT / "output" / "lesson04_synthetic" / "SYNTHETIC_usage_2025.csv"
    usage = load_usage_csv(usage_path)
    reviewed_bills = [calculate_month(month, reviewed_plan) for month in usage.months]
    print("Hypothetical subtotal with synthetic usage, not a household forecast:")
    for bill in reviewed_bills:
        print(bill.month, bill.kwh, bill.total_usd)
    print("Annual energy/base/delivery subtotal USD:", sum(b.total_usd for b in reviewed_bills))
    print(SCENARIO_ASSUMPTIONS)

BLOCKED: resolve the listed evidence and review requirements first.


## 7. Save a readiness report
Save the reviewed input snapshot and current blockers. This does not approve the
plan, overwrite the extraction or change Notebook 03. Rerun Section 6 before this
cell after any edits. The report includes costs only when the guarded calculation ran.

In [ ]:
if PricingReview.model_validate_json(review_path.read_text(encoding="utf-8")) != review:
    raise RuntimeError("Review changed; rerun Sections 5 and 6 before saving.")
readiness_report = {
    "status": "blocked" if blockers else "reviewed_for_limited_scenario",
    "scenario_assumptions": SCENARIO_ASSUMPTIONS,
    "review": review.model_dump(mode="json"),
    "blockers": blockers,
    "synthetic_usage": True,
    "bills": [b.model_dump(mode="json") for b in reviewed_bills] if reviewed_bills is not None else None,
}
readiness_path = review_dir / "Reliant_readiness.json"
readiness_path.write_text(json.dumps(readiness_report, indent=2), encoding="utf-8")
print("Saved:", readiness_path)

Saved: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\review\Reliant_readiness.json


## What you learned
A cited extraction is a set of candidate terms. A reviewed pricing input additionally
needs units, applicable conditions, completeness and a human review record.
Our gate prevents missing values from becoming plausible-looking costs.

**Next action:** inspect the blockers and determine which require another document.
Do not invent missing fees. After supporting documents are available, we can complete
the review and connect compatible reviewed plans to the supervisor's comparison step.
That real-plan ranking connection is still separate from this single-plan adapter.